In [1]:

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime
import warnings
import os



In [2]:
# Load splits
train_data = pd.read_pickle('dataset/train_data_final.pkl')
val_data_masked = pd.read_pickle('dataset/val_data_masked.pkl')
test_data_masked = pd.read_pickle('dataset/test_data_masked.pkl')

In [4]:
# Load ground truth
val_ground_truth = pd.read_pickle('dataset/val_ground_truth.pkl')
test_ground_truth = pd.read_pickle('dataset/test_ground_truth.pkl')

# Load mask indicators
val_mask_indicator = pd.read_pickle('dataset/val_mask_indices.pkl')
test_mask_indicator = pd.read_pickle('dataset/test_mask_indices.pkl')





In [5]:
print(f"\n✓ Loaded all data:")
print(f"  • Train: {train_data.shape}")
print(f"  • Val (masked): {val_data_masked.shape}")
print(f"  • Test (masked): {test_data_masked.shape}")


✓ Loaded all data:
  • Train: (143459, 59)
  • Val (masked): (30741, 59)
  • Test (masked): (30742, 59)


In [6]:
# Get features that were masked
features_to_impute = val_ground_truth.columns.tolist()
print(f"\n  • Features to impute: {len(features_to_impute)}")


  • Features to impute: 45


In [7]:
# Get all columns
all_columns = train_data.columns.tolist()

# Identify non-impute features
non_impute_features = [col for col in all_columns if col not in features_to_impute]

print(f"\n FEATURE SPLIT:")
print(f"  • Features to impute: {len(features_to_impute)}")
print(f"  • Features to keep as-is: {len(non_impute_features)}")


 FEATURE SPLIT:
  • Features to impute: 45
  • Features to keep as-is: 14


In [10]:
from sklearn.preprocessing import MinMaxScaler, RobustScaler

In [11]:

scaler = MinMaxScaler()

# Fit on training data
train_values = train_data[features_to_impute].values
scaler.fit(train_values)

print("Fitted MinMaxScaler on training data")

# Transform all datasets
train_scaled = scaler.transform(train_values)
val_scaled = scaler.transform(val_data_masked[features_to_impute].values)
test_scaled = scaler.transform(test_data_masked[features_to_impute].values)

# Also scale ground truth (for loss calculation)
val_gt_scaled = scaler.transform(val_ground_truth.fillna(0).values)  # fillna for transform
test_gt_scaled = scaler.transform(test_ground_truth.fillna(0).values)


Fitted MinMaxScaler on training data


In [12]:


# Check the actual scale of your data AFTER RobustScaler
print("\nChecking scaled data statistics:")
print(f"{'Statistic':<20s} {'Value':<20s}")
print("─"*45)

train_scaled_stats = {
    'Min': train_scaled.min(),
    'Max': train_scaled.max(),
    'Mean': np.nanmean(train_scaled),
    'Std': np.nanstd(train_scaled),
    'Median': np.nanmedian(train_scaled),
    '99th percentile': np.nanpercentile(train_scaled, 99),
    '1st percentile': np.nanpercentile(train_scaled, 1)
}

for stat, value in train_scaled_stats.items():
    print(f"{stat:<20s} {value:<20.4f}")

print(f"\n  If Max/Min are > 1000, scaling didn't work well!")

# Check individual features
print(f"\n Top 10 features with largest scaled values:")
feature_max_vals = []
for i, feat in enumerate(features_to_impute):
    col_data = train_scaled[:, i]
    max_abs = np.nanmax(np.abs(col_data))
    feature_max_vals.append((feat, max_abs))

feature_max_vals.sort(key=lambda x: x[1], reverse=True)

print(f"{'Feature':<50s} {'Max |value|':<15s}")
print("─"*70)
for feat, max_val in feature_max_vals[:10]:
    print(f"{feat:<50s} {max_val:<15.2e}")



Checking scaled data statistics:
Statistic            Value               
─────────────────────────────────────────────
Min                  nan                 
Max                  nan                 
Mean                 0.4724              
Std                  0.3111              
Median               0.4867              
99th percentile      1.0000              
1st percentile       0.0000              

  If Max/Min are > 1000, scaling didn't work well!

 Top 10 features with largest scaled values:
Feature                                            Max |value|    
──────────────────────────────────────────────────────────────────────
ping_ms                                            1.00e+00       
datarate                                           1.00e+00       
jitter                                             1.00e+00       
Latitude                                           1.00e+00       
Longitude                                          1.00e+00       
speed_kmh    

In [13]:

def clip_outliers_per_feature(data, lower_pct=0.5, upper_pct=99.5):
    """
    Clip outliers per feature based on percentiles.
    Handles NaN values properly.
    """
    data_clipped = data.copy()
    
    for i in range(data.shape[1]):
        col = data[:, i]
        
        # Get valid (non-NaN) values
        valid_mask = ~np.isnan(col)
        
        if valid_mask.sum() > 0:
            valid_values = col[valid_mask]
            
            # Calculate percentiles
            lower_bound = np.percentile(valid_values, lower_pct)
            upper_bound = np.percentile(valid_values, upper_pct)
            
            # Clip only valid values
            col[valid_mask] = np.clip(valid_values, lower_bound, upper_bound)
            data_clipped[:, i] = col
    
    return data_clipped

In [14]:

train_values = train_data[features_to_impute].values

In [15]:
train_values_clipped = clip_outliers_per_feature(train_values, lower_pct=0.5, upper_pct=99.5)


In [16]:
for i, feat in enumerate(features_to_impute[:10]):  # Show first 10
    original_max = np.nanmax(np.abs(train_values[:, i]))
    clipped_max = np.nanmax(np.abs(train_values_clipped[:, i]))
    
    if original_max > 1000:  # Only show features that were clipped significantly
        print(f"  • {feat[:45]:45s}: {original_max:>12.2e} → {clipped_max:>12.2e}")

In [17]:
scaler = MinMaxScaler()
scaler.fit(train_values_clipped)

# Transform all datasets (with outlier clipping)
train_scaled = scaler.transform(train_values_clipped)

val_values_clipped = clip_outliers_per_feature(val_data_masked[features_to_impute].values)
val_scaled = scaler.transform(val_values_clipped)

test_values_clipped = clip_outliers_per_feature(test_data_masked[features_to_impute].values)
test_scaled = scaler.transform(test_values_clipped)

In [18]:

new_stats = {
    'Min': np.nanmin(train_scaled),
    'Max': np.nanmax(train_scaled),
    'Mean': np.nanmean(train_scaled),
    'Std': np.nanstd(train_scaled),
    '1st percentile': np.nanpercentile(train_scaled, 1),
    '99th percentile': np.nanpercentile(train_scaled, 99)
}

print(f"{'Statistic':<20s} {'Value':<15s}")
print("─"*40)
for stat, value in new_stats.items():
    print(f"{stat:<20s} {value:<15.4f}")

print(f"\n✓ Values should now be in reasonable range (-5 to +5)")

# Check max values per feature
print(f"\nTop 5 features by max scaled value:")
feature_max_scaled = []
for i, feat in enumerate(features_to_impute):
    max_val = np.nanmax(np.abs(train_scaled[:, i]))
    feature_max_scaled.append((feat, max_val))

feature_max_scaled.sort(key=lambda x: x[1], reverse=True)

for feat, max_val in feature_max_scaled[:5]:
    print(f"  • {feat[:45]:45s}: {max_val:>10.4f}")


Statistic            Value          
────────────────────────────────────────
Min                  0.0000         
Max                  1.0000         
Mean                 0.4623         
Std                  0.3187         
1st percentile       0.0000         
99th percentile      1.0000         

✓ Values should now be in reasonable range (-5 to +5)

Top 5 features by max scaled value:
  • Traffic Distance                             :     1.0000
  • Pos in Ref Round                             :     1.0000
  • PCell_RSRP_2                                 :     1.0000
  • ping_ms                                      :     1.0000
  • Latitude                                     :     1.0000


In [16]:
from sklearn.preprocessing import StandardScaler

# Fit scaler on training data
#scaler = StandardScaler()
#train_impute_array = train_data[features_to_impute].values
#scaler.fit(train_impute_array)


# Transform all sets
#train_scaled = scaler.transform(train_impute_array)
#val_scaled = scaler.transform(val_data_masked[features_to_impute].values)
#test_scaled = scaler.transform(test_data_masked[features_to_impute].values)



In [19]:
import numpy as np
import pandas as pd
from sklearn.experimental import enable_iterative_imputer 
from sklearn.impute import KNNImputer, IterativeImputer, SimpleImputer
from sklearn.metrics import mean_squared_error, mean_absolute_error

In [20]:

def evaluate_imputation(imputed_data, ground_truth, mask_indicator, method_name, split_name):
  
    results = []
    all_errors_squared = []
    all_errors_abs = []
    
    for col in ground_truth.columns:
        # Get mask for this feature
        mask = mask_indicator[col]
        
        # Count masked values
        n_masked = mask.sum()
        
        if n_masked == 0:
            continue
        
        # Get ground truth values (only non-NaN)
        true_vals = ground_truth.loc[mask, col].dropna()
        
        
        # Get predicted values at same positions
        pred_vals = imputed_data.loc[true_vals.index, col]
        
        # Calculate errors
        errors = true_vals - pred_vals
        errors_squared = errors ** 2
        errors_abs = errors.abs()
        
        rmse = np.sqrt(errors_squared.mean())
        mae = errors_abs.mean()
        
        # Store results
        results.append({
            'feature': col,
            'n_masked': len(true_vals),
            'RMSE': rmse,
            'MAE': mae
        })
        
        # Collect for overall metrics
        all_errors_squared.extend(errors_squared.values)
        all_errors_abs.extend(errors_abs.values)
    
    # Overall metrics
    overall_rmse = np.sqrt(np.mean(all_errors_squared))
    overall_mae = np.mean(all_errors_abs)
    
    results_df = pd.DataFrame(results)
    
    return results_df, overall_rmse, overall_mae

print("\n EVALUATING METHODS...")



 EVALUATING METHODS...


In [21]:
#forward/backward fill

def forward_backward_fill_imputation(data, features_to_impute):
    
    data_imputed = data.copy()
    
    # Apply forward fill then backward fill to impute features only
    for col in features_to_impute:
        if col in data_imputed.columns:
            # Forward fill
            data_imputed[col] = data_imputed[col].fillna(method='ffill')
            # Backward fill
            data_imputed[col] = data_imputed[col].fillna(method='bfill')
    
    return data_imputed

print("\n Applying Forward/Backward Fill...")

# Validation
print("\n  Processing validation set...")
val_ffill = forward_backward_fill_imputation(val_data_masked, features_to_impute)

# Test
print("  Processing test set...")
test_ffill = forward_backward_fill_imputation(test_data_masked, features_to_impute)




 Applying Forward/Backward Fill...

  Processing validation set...
  Processing test set...


C:\Users\Aditya\AppData\Local\Temp\ipykernel_12964\178238238.py:11: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  data_imputed[col] = data_imputed[col].fillna(method='ffill')
C:\Users\Aditya\AppData\Local\Temp\ipykernel_12964\178238238.py:13: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  data_imputed[col] = data_imputed[col].fillna(method='bfill')


In [22]:
# Check remaining missing
val_remaining = val_ffill[features_to_impute].isnull().sum().sum()
test_remaining = test_ffill[features_to_impute].isnull().sum().sum()

In [ ]:
print(f"    Validation: {val_remaining:,} (should be 0 or very small)")
print(f"    Test: {test_remaining:,} (should be 0 or very small)")

  • Validation: 0 (should be 0 or very small)
  • Test: 0 (should be 0 or very small)


In [24]:
if val_remaining > 0 or test_remaining > 0:
    print(f"  Fill with column mean")
    
    # Fallback: fill with mean
    for col in features_to_impute:
        if val_ffill[col].isnull().sum() > 0:
            col_mean = train_data[col].mean()
            val_ffill[col].fillna(col_mean, inplace=True)
        
        if test_ffill[col].isnull().sum() > 0:
            col_mean = train_data[col].mean()
            test_ffill[col].fillna(col_mean, inplace=True)

In [25]:

method_name = 'Forward/Backward Fill'


# Validation
val_results, val_rmse, val_mae = evaluate_imputation(
    val_ffill, val_ground_truth, val_mask_indicator,
    method_name, 'Validation'
)

# Test
test_results, test_rmse, test_mae = evaluate_imputation(
    test_ffill, test_ground_truth, test_mask_indicator,
    method_name, 'Test'
)

print(f"\n  VALIDATION:")
print(f"    • Overall RMSE: {val_rmse:.4f}")
print(f"    • Overall MAE:  {val_mae:.4f}")
print(f"    • Features evaluated: {len(val_results)}")

print(f"\n  TEST:")
print(f"    • Overall RMSE: {test_rmse:.4f}")
print(f"    • Overall MAE:  {test_mae:.4f}")
print(f"    • Features evaluated: {len(test_results)}")
    


  VALIDATION:
    • Overall RMSE: 37080.5747
    • Overall MAE:  166.8715
    • Features evaluated: 45

  TEST:
    • Overall RMSE: 1426.2942
    • Overall MAE:  51.0826
    • Features evaluated: 45


In [26]:

# Create KNN imputer
knn_imputer = KNNImputer(n_neighbors=5, weights='uniform')


knn_imputer.fit(train_scaled)

# Transform validation (on scaled data)
print("  Imputing validation set...")
val_knn_scaled = knn_imputer.transform(val_scaled)

# Inverse transform back to original scale
val_knn_original = scaler.inverse_transform(val_knn_scaled)

# Create full dataframe
val_knn = val_data_masked.copy()
val_knn[features_to_impute] = val_knn_original

# Transform test (on scaled data)
print("  Imputing test set...")
test_knn_scaled = knn_imputer.transform(test_scaled)

# Inverse transform back to original scale
test_knn_original = scaler.inverse_transform(test_knn_scaled)

# Create full dataframe
test_knn = test_data_masked.copy()
test_knn[features_to_impute] = test_knn_original


  Imputing validation set...
  Imputing test set...


In [27]:

method_name = 'KNN Imputation'


# Validation
val_results, val_rmse, val_mae = evaluate_imputation(
    val_knn, val_ground_truth, val_mask_indicator,
    method_name, 'Validation'
)

# Test
test_results, test_rmse, test_mae = evaluate_imputation(
    test_knn, test_ground_truth, test_mask_indicator,
    method_name, 'Test'
)

print(f"\n  VALIDATION:")
print(f"      Overall RMSE: {val_rmse:.4f}")
print(f"      Overall MAE:  {val_mae:.4f}")
print(f"      Features evaluated: {len(val_results)}")

print(f"\n  TEST:")
print(f"      Overall RMSE: {test_rmse:.4f}")
print(f"      Overall MAE:  {test_mae:.4f}")
print(f"      Features evaluated: {len(test_results)}")
    


  VALIDATION:
      Overall RMSE: 37060.2934
      Overall MAE:  123.4332
      Features evaluated: 45

  TEST:
      Overall RMSE: 107.3816
      Overall MAE:  15.9248
      Features evaluated: 45


In [28]:

# Store all results
all_results = {}

# Methods to evaluate
methods = [
    ('Forward/Backward Fill', val_ffill, test_ffill),
    ('KNN (k=5)', val_knn, test_knn)
]

for method_name, val_imputed, test_imputed in methods:
    print(f"EVALUATING: {method_name}")
    
    # Validation
    val_results, val_rmse, val_mae = evaluate_imputation(
        val_imputed, val_ground_truth, val_mask_indicator,
        method_name, 'Validation'
    )
    
    # Test
    test_results, test_rmse, test_mae = evaluate_imputation(
        test_imputed, test_ground_truth, test_mask_indicator,
        method_name, 'Test'
    )
    
    print(f"\n  VALIDATION:")
    print(f"      Overall RMSE: {val_rmse:.4f}")
    print(f"      Overall MAE:  {val_mae:.4f}")
    print(f"      Features evaluated: {len(val_results)}")
    
    print(f"\n  TEST:")
    print(f"      Overall RMSE: {test_rmse:.4f}")
    print(f"      Overall MAE:  {test_mae:.4f}")
    print(f"      Features evaluated: {len(test_results)}")
    
    # Store results
    all_results[method_name] = {
        'val_results': val_results,
        'test_results': test_results,
        'val_rmse': val_rmse,
        'val_mae': val_mae,
        'test_rmse': test_rmse,
        'test_mae': test_mae
    }

EVALUATING: Forward/Backward Fill

  VALIDATION:
      Overall RMSE: 37080.5747
      Overall MAE:  166.8715
      Features evaluated: 45

  TEST:
      Overall RMSE: 1426.2942
      Overall MAE:  51.0826
      Features evaluated: 45
EVALUATING: KNN (k=5)

  VALIDATION:
      Overall RMSE: 37060.2934
      Overall MAE:  123.4332
      Features evaluated: 45

  TEST:
      Overall RMSE: 107.3816
      Overall MAE:  15.9248
      Features evaluated: 45
